# Homework – Week 4: Digital Twin Synchronization
**Course:** Simulations – Digital Twin  
**Team:** A – Indoor $\text{PM}_{2.5}$ (Weslaco Residence & TCEQ Meteorological Data)

## Objective
Extend the existing indoor air quality mass-balance simulation into a chronological replay-mode Digital Twin using historical indoor and outdoor observations.

---

## Task 1: Prepare Observation Data
* **Dataset:** Mendeley Data (`doi: 10.17632/yk2pz4hcb5.2`) – 1-hour average indoor and outdoor $\text{PM}_{2.5}$ ($\mu\text{g/m}^3$) and relative humidity ($\text{RH}$, %) from Home-1 (Weslaco), merged with resultant wind speed ($\text{RWS}$, m/s) from TCEQ CAMS C1023 (Harlingen).
* **Data Quality Checks:**
  1. Parse and align timestamps (`datetime`) in strict chronological order.
  2. Check for and remove missing values (`NaN`) and duplicate timestamps.
  3. Filter out any invalid (negative) physical measurements.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

# 1. Load datasets (use local path or Mendeley direct URLs)
weslaco_csv = "../data/Home1_Weslaco_Indoor_Outdoor_updated.csv"
wind_csv = "../data/TCEQ_CAMSdata_WindRose.csv"

df_w = pd.read_csv(weslaco_csv, low_memory=False)
df_wind = pd.read_csv(wind_csv, low_memory=False)

# Extract 1-hour Indoor, Outdoor, and Wind tables (row 0 contains sub-headers)
indoor_1h = df_w.iloc[1:, 18:23].dropna(how='all').copy()
indoor_1h.columns = ['datetime', 'T_in', 'rh_in', 'raw_pm25_in', 'pm25_in']

outdoor_1h = df_w.iloc[1:, 26:31].dropna(how='all').copy()
outdoor_1h.columns = ['datetime', 'T_out', 'rh', 'raw_pm25_out', 'pm25']

wind_1h = df_wind.iloc[1:, 5:8].dropna(how='all').copy()
wind_1h.columns = ['datetime', 'wind_dir', 'wind']

# Convert timestamps and numeric columns
for d in [indoor_1h, outdoor_1h, wind_1h]:
    d['datetime'] = pd.to_datetime(d['datetime'].astype(str).str.strip())
    for col in d.columns:
        if col != 'datetime':
            d[col] = pd.to_numeric(d[col], errors='coerce')

# Merge datasets on timestamp
df = outdoor_1h[['datetime', 'rh', 'pm25']].merge(
    indoor_1h[['datetime', 'rh_in', 'pm25_in']], on='datetime', how='inner'
).merge(
    wind_1h[['datetime', 'wind']], on='datetime', how='inner'
)

# Check timestamps, missing values, and invalid records
print(f"Initial merged records: {len(df)}")
print(f"Missing values per column:\n{df[['datetime', 'wind', 'rh', 'pm25', 'pm25_in']].isna().sum()}\n")

# Remove missing values, duplicates, and any invalid (< 0) records, then sort chronologically
df = df.dropna().drop_duplicates(subset=['datetime'], keep='first')
df = df[(df['pm25'] >= 0) & (df['pm25_in'] >= 0) & (df['wind'] >= 0) & (df['rh'] >= 0)]
df = df.sort_values('datetime').reset_index(drop=True)

print(f"Cleaned chronological dataset shape: {df.shape}")
print(f"Chronological order verified: {df['datetime'].is_monotonic_increasing}")

# Select 24-hour evaluation window (2024-06-16)
df_day = df[:24].copy()
display(df_day[['datetime', 'wind', 'rh', 'pm25', 'pm25_in']].head())